In [1]:
!pip install groq

In [2]:
from groq import Groq

cl=Groq(api_key="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")

In [3]:

import json

# ---------------------------------------
# 1. PYTHON TEACHING TOOLS
# ---------------------------------------

def teach_python(topic: str, level: str):
    return {
        "task": "teach",
        "topic": topic,
        "level": level,
        "instructions": """
        Explain the concept with:
        1. Definition
        2. Detailed theory
        3. How it works internally
        4. Syntax
        5. Simple Python example
        6. Output
        7. Real-world application
        8. Common mistakes
        9. Interview questions
        """
    }


def generate_program(topic: str, requirements: str):
    return {
        "task": "generate_program",
        "topic": topic,
        "requirements": requirements,
        "instructions": """
        Generate a complete runnable Python program.
        Explain each important line.
        Show expected output.
        Include a real-world example.
        """
    }


def create_quiz(topic: str, difficulty: str):
    return {
        "task": "quiz",
        "topic": topic,
        "difficulty": difficulty,
        "instructions": """
        Generate 5 Python questions.
        Ask one question at a time.
        Wait for the student's answer.
        Evaluate the answer and explain the correct concept.
        """
    }


def review_code(code: str, goal: str):
    return {
        "task": "review_code",
        "code": code,
        "goal": goal,
        "instructions": """
        Review the student's Python code.
        Identify syntax errors and logical errors.
        Explain why errors occur.
        Give corrected code.
        Explain the improvements.
        """
    }


# ---------------------------------------
# 2. DEFINE AGENT TOOLS FOR GROQ
# ---------------------------------------

agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "teach_python",
            "description": "Teach and explain a Python concept in detail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "level": {
                        "type": "string",
                        "enum": ["beginner", "intermediate", "advanced"]
                    }
                },
                "required": ["topic", "level"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_program",
            "description": "Generate a Python program for a requested task.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "requirements": {"type": "string"}
                },
                "required": ["topic", "requirements"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_quiz",
            "description": "Create a Python learning quiz.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "difficulty": {
                        "type": "string",
                        "enum": ["beginner", "intermediate", "advanced"]
                    }
                },
                "required": ["topic", "difficulty"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "review_code",
            "description": "Review and correct a student's Python code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string"},
                    "goal": {"type": "string"}
                },
                "required": ["code", "goal"],
                "additionalProperties": False
            }
        }
    }
]


# ---------------------------------------
# 3. TOOL EXECUTION
# ---------------------------------------

def execute_tool(name, args):

    if name == "teach_python":
        return teach_python(**args)

    elif name == "generate_program":
        return generate_program(**args)

    elif name == "create_quiz":
        return create_quiz(**args)

    elif name == "review_code":
        return review_code(**args)

    return {"error": "Unknown tool"}


# ---------------------------------------
# 4. PYTHON TEACHING AGENT
# ---------------------------------------

def python_teaching_agent():

    messages = [
        {
            "role": "system",
            "content": """
            You are a professional Python Programming Teacher.

            Teach students in a friendly and understandable way.

            Your teaching responsibilities:

            1. Explain Python concepts in depth.
            2. Generate practical Python programs.
            3. Conduct quizzes one question at a time.
            4. Evaluate student answers.
            5. Review and correct Python code.
            6. Explain concepts using real-world examples.
            7. Show code output.
            8. Ask the student whether they understand before
               moving to the next quiz question.

            Use the available tools when appropriate.

            Do not execute student code.
            Explain concepts accurately.
            """
        }
    ]

    print("Python Teaching AI Agent Started!")
    print("Ask me anything about Python.")
    print("Type exit to stop.")

    while True:

        user_input = input("\nStudent: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Python Teaching Agent stopped.")
            break

        messages = messages + [
            {"role": "user", "content": user_input}
        ]

        # Ask Groq to understand the request
        response = cl.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=messages,
            tools=agent_tools,
            tool_choice="auto",
            temperature=0
        )

        assistant = response.choices[0].message

        # Handle tool calls
        while assistant.tool_calls:

            messages = messages + [
                assistant.model_dump(exclude_none=True)
            ]

            for tool_call in assistant.tool_calls:

                tool_name = tool_call.function.name

                arguments = json.loads(
                    tool_call.function.arguments
                )

                print("\nAgent selected tool:", tool_name)

                result = execute_tool(
                    tool_name,
                    arguments
                )

                messages = messages + [
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(result)
                    }
                ]

            # Ask Groq to prepare the teaching response
            response = cl.chat.completions.create(
                model="qwen/qwen3.8-27b",
                messages=messages,
                tools=agent_tools,
                tool_choice="auto",
                temperature=0
            )

            assistant = response.choices[0].message

        answer = assistant.content or ""

        messages = messages + [
            {"role": "assistant", "content": answer}
        ]

        print("\nPython Teacher:", answer)


# ---------------------------------------
# 5. START THE AGENT
# ---------------------------------------

python_teaching_agent()


Python Teaching AI Agent Started!
Ask me anything about Python.
Type exit to stop.

Student: who created python and why it gets the name python

Python Teacher: Great question! Let me share the story behind Python's creation and its name.

## Who Created Python?

**Guido van Rossum**, a Dutch programmer, created Python. He started working on it in **December 1989** while at the **Stichting Mathematisch Centrum** (CWI) in the Netherlands. The first public release, Python 0.9.0, came out in **February 1991**.

Guido designed Python with a clear philosophy:
- **Readability first** – code should look clean and be easy to understand
- **Simplicity** – fewer ways to do the same thing, so less confusion
- **Practicality** – "batteries included" (a rich standard library)

## Why Is It Called "Python"?

This is a fun one! The name has **nothing to do with the snake** 🐍 (at least, not originally).

Guido was a big fan of the British comedy group **Monty Python** (famous for *Monty Python's Flyin

In [6]:
lt="""
## Who Created Python?

**Guido van Rossum**, a Dutch programmer, created Python. He started working on it in **December 1989** while at the **Stichting Mathematisch Centrum** (CWI) in the Netherlands. The first public release, Python 0.9.0, came out in **February 1991**.

Guido designed Python with a clear philosophy:
- **Readability first** – code should look clean and be easy to understand
- **Simplicity** – fewer ways to do the same thing, so less confusion
- **Practicality** – "batteries included" (a rich standard library)

## Why Is It Called "Python"?

This is a fun one! The name has **nothing to do with the snake** 🐍 (at least, not originally).

Guido was a big fan of the British comedy group **Monty Python** (famous for *Monty Python's Flying Circus*). He wanted a name that was:
- **Short**
- **Unique**
- **Slightly mysterious**

So he named his new programming language **"Python"** after the comedy troupe. The snake logo that became Python's mascot came later as a playful nod to the name.

> Fun fact: Guido was the "Benevolent Dictator For Life" (BDFL) of the Python project for many years, guiding its development until he stepped down in 2020.

---

Would you like me to:
1. **Teach you a Python concept** (like variables, loops, or functions)?
2. **Generate a small Python program** to get you started?
3. **Quiz you** on Python basics?

Just let me know what you'd like to explore next! 😊

Student: exit
Python Teaching Agent stopped.
"""

In [8]:
response = cl.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {
            "role": "system",
            "content": "Act as a judge to evaluate the answers and give some ratings for that."
        },
        {
            "role": "user",
            "content":lt
        }
    ]
)

print(response.choices[0].message.content)

As the judge, I will evaluate the response provided by the "Python Teaching Agent" based on accuracy, clarity, tone, and completeness, assuming the context was a user asking, "Who created Python and why is it called that?" or simply ending the conversation.

### Evaluation

**1. Factual Accuracy (10/10)**
*   **Creator:** Correctly identified as Guido van Rossum.
*   **Timeline:** Correctly noted the start in Dec 1989 and first public release in Feb 1991.
*   **Origin of Name:** Correctly attributed to the comedy group Monty Python, not the snake.
*   **BDFL Status:** Correctly stated he stepped down in 2020.
*   **Philosophy:** Accurately summarized PEP 20 (The Zen of Python) principles (readability, simplicity, practicality).

**2. Clarity and Structure (10/10)**
*   The response is well-organized with clear headings.
*   Bullet points are used effectively to list key facts about the name and philosophy.
*   The "Fun fact" adds engaging context without cluttering the core information